## Section C: device.csv

Initialization - same steps as in "src\entry_and_logon.ipynb"

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CERT Analysis") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.executorEnv.SPARK_LOCAL_IP", "127.0.0.1") \
    .getOrCreate()


In [3]:
# utils.py contains all helpers
from helpers import code_quality, parse_date, letters_cleaning

In [4]:
# reading device.csv 
device_raw = spark.read.csv("../data/device.csv", header=True, inferSchema=True)

In [5]:
device_raw.show(3)

+--------------------+-------------------+-------+-------+----------+
|                  id|               date|   user|     pc|  activity|
+--------------------+-------------------+-------+-------+----------+
|{P1I9-X2UT34LR-64...|01/02/2010 05:15:32|WCR0044|PC-9174|   Connect|
|{S2D2-V1NT65KD-81...|01/02/2010 05:17:09|WCR0044|PC-9174|Disconnect|
|{G5Y9-L4GP40WG-03...|01/02/2010 06:39:10|WCR0044|PC-5494|   Connect|
+--------------------+-------------------+-------+-------+----------+
only showing top 3 rows


File `device.csv` contains: additional/disconnected USB/external devices. <br>
Columns: ID, date, user, pc, activity (Connect/Disconnect) <br>
Signal exfiltration significance for GNN - copying to USB is one of the main malicious insider patterns

In [6]:
# quality report
code_quality(device_raw, "device.csv")

Data quality for: device.csv
Number of records: 437168, number of columns: 5
Missing df: [Row(id=0, date=0, user=0, pc=0, activity=0)]

Duplicates: 0 (0.00%)


{'rows': 437168,
 'columns': 5,
 'duplicates': 0,
 'missing': DataFrame[id: bigint, date: bigint, user: bigint, pc: bigint, activity: bigint]}

In [7]:
# Cleaning
device = device_raw \
    .dropDuplicates() \
    .filter(F.col("user").isNotNull()) \
    .filter(F.col("activity").isin("Connect", "Disconnect"))

In [8]:
# parsing date
device = parse_date(device_raw)
device.show()

+--------------------+-------------------+-------+-------+----------+-------------------+----+-----+---+----+---------------+-----------------+
|                  id|               date|   user|     pc|  activity|          timestamp|year|month|day|hour|day_of_the_week|not_working_hours|
+--------------------+-------------------+-------+-------+----------+-------------------+----+-----+---+----+---------------+-----------------+
|{P1I9-X2UT34LR-64...|01/02/2010 05:15:32|WCR0044|PC-9174|   Connect|2010-01-02 05:15:32|2010|    1|  2|   5|              7|             true|
|{S2D2-V1NT65KD-81...|01/02/2010 05:17:09|WCR0044|PC-9174|Disconnect|2010-01-02 05:17:09|2010|    1|  2|   5|              7|             true|
|{G5Y9-L4GP40WG-03...|01/02/2010 06:39:10|WCR0044|PC-5494|   Connect|2010-01-02 06:39:10|2010|    1|  2|   6|              7|             true|
|{A4Q5-A1FR45TY-95...|01/02/2010 06:40:30|WCR0044|PC-5494|Disconnect|2010-01-02 06:40:30|2010|    1|  2|   6|              7|           

In [9]:
# Top 10 users with biggest amount of USB Connections
device.filter(F.col("activity")=="Connect").groupBy("user").count().orderBy(F.desc("count")).show(10)

+-------+-----+
|   user|count|
+-------+-----+
|SPW0142| 4039|
|BHV0556| 3983|
|LGS0292| 3882|
|GBM0007| 3780|
|FLC0370| 3756|
|KBD0201| 3756|
|OSF0896| 3754|
|TDS0246| 3129|
|MDA0815| 3042|
|DCR0089| 3039|
+-------+-----+
only showing top 10 rows


In [10]:
# USB Connections after working hours
device.filter(F.col("activity")=="Connect").groupBy("not_working_hours").count().show()

+-----------------+------+
|not_working_hours| count|
+-----------------+------+
|             true| 15491|
|            false|203745|
+-----------------+------+



In [11]:
# device features - soon to be needed when creating statistical daily user's behaviour
device_features = device.filter(F.col("activity") == "Connect") \
    .groupBy("user", "year", "month", "day") \
    .agg(
        F.count("*").alias("usb_connections"),
        F.sum(F.col("not_working_hours").cast("int")).alias("usb_nightly"),
        F.countDistinct("pc").alias("pcs_usb")
    )

In [12]:
device_features.show()

+-------+----+-----+---+---------------+-----------+-------+
|   user|year|month|day|usb_connections|usb_nightly|pcs_usb|
+-------+----+-----+---+---------------+-----------+-------+
|BJW0369|2010|    2| 25|              6|          3|      6|
|PWS0219|2010|    2| 25|              7|          3|      6|
|SAM0359|2010|    2| 21|              4|          4|      4|
|CJM0243|2010|    1| 27|              7|          0|      1|
|PWS0219|2010|    2| 26|              5|          3|      4|
|SDH0026|2010|    2| 19|              8|          1|      1|
|SRF0199|2010|    1| 18|              1|          0|      1|
|PHJ0041|2010|    1| 10|              4|          1|      2|
|LKH0051|2010|    1| 25|              1|          0|      1|
|NSL0963|2010|    2| 10|              3|          0|      1|
|EYE0188|2010|    1| 27|              1|          0|      1|
|KBD0201|2010|    2|  4|              8|          0|      1|
|CIH0704|2010|    1| 15|              3|          0|      1|
|ATB0075|2010|    2|  1|

In [ ]:
import os

logon_features_pd = device_features.toPandas()
logon_pd = device.toPandas()

logon_features_pd.to_parquet(
    "your_link",
    index=False
)

logon_pd.to_parquet(
    "your_link",
    index=False
)

print("saved, finally -_-")

saved, finally -_-


In [14]:
spark.stop()